In [1]:
import pandas as pd
import numpy as np
import re
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pandas.errors import SettingWithCopyWarning
#from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("ignore", SettingWithCopyWarning)

## Load Datasets


In [2]:
# BoardGameGeek dataset
bgg_dataset = pd.read_csv("bgg_dataset.csv", sep=';')
bgg_dataset.head(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,"8,79",1,"3,86",68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,"8,61",2,"2,84",65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,"8,66",3,"3,91",28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,"8,43",4,"3,24",87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,"8,70",5,"4,22",16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


In [3]:
# Product/edition dataset
products_dataset = pd.read_csv('products_with_edition.csv')
products_dataset.head(5)

,productid,objecttype,objectid,producthref,price,price_in_usd,currency,currencystring,currencysymbol,condition,...,quantity,userId,username,user_country,user_city,version_name,version_href,game_name,game_href,linkeduserGeekMarket
0,3870857,thing,1,/market/product/3870857,30.00,30.00,USD,U.S. dollars,$,likenew,...,1,2954044,LeBao,United States,Tallahassee,Die Macher (Multilingual edition 2006),/boardgameversion/24534/multilingual-edition-2006,Die Macher,/boardgame/1/die-macher,NaN
1,3866054,thing,1,/market/product/3866054,180.00,180.00,USD,U.S. dollars,$,new,...,1,3677448,CarsonChandler,United States,Christiansburg,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
2,3854476,thing,1,/market/product/3854476,41.99,41.99,USD,U.S. dollars,$,likenew,...,1,215794,Eric Engelmann,United States,Rockville,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
3,3853310,thing,1,/market/product/3853310,60.00,66.00,EUR,Euros,€,new,...,1,1613268,JillySilly,Belgium,NaN,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
4,3845442,thing,1,/market/product/3845442,50.00,50.00,USD,U.S. dollars,$,verygood,...,1,38240,caesarmom,United States,Lorton,Die Macher (German-only second edition),/boardgameversion/24939/german-only-second-edi...,Die Macher,/boardgame/1/die-macher,NaN


## Currency conversion in product_dataset

In [4]:
def convert_to_usd(amount, currency, rates):
    currency = str(currency).upper()
    if currency not in rates or pd.isna(amount):
        return None
    return round(float(amount) * rates[currency], 2)

rates = {
    'USD': 1.0,
    'EUR': 1.10,
    'GBP': 1.30,
    'AUD': 0.65,
    'CAD': 0.75,
    'CHF': 1.05,
    'SGD': 0.73,
    'SEK': 0.10,
    'PLN': 0.24,
    'HUF': 0.0027,
    'MXN': 0.057,
    'NZD': 0.60,
    'JPY': 0.0073,
    'CNY': 0.14,
    'BRL': 0.19,
    'CZK': 0.042,
    'MYR': 0.21,
}

products_dataset['currency'] = products_dataset['currency'].str.upper()

products_dataset['price'] = products_dataset['price'] * products_dataset['currency'].map(rates)

products_dataset['currency'] = 'USD'

products_dataset['price'] = products_dataset['price'].round(2)
products_dataset

,productid,objecttype,objectid,producthref,price,price_in_usd,currency,currencystring,currencysymbol,condition,...,quantity,userId,username,user_country,user_city,version_name,version_href,game_name,game_href,linkeduserGeekMarket
0,3870857,thing,1,/market/product/3870857,30.00,30.00,USD,U.S. dollars,$,likenew,...,1,2954044,LeBao,United States,Tallahassee,Die Macher (Multilingual edition 2006),/boardgameversion/24534/multilingual-edition-2006,Die Macher,/boardgame/1/die-macher,NaN
1,3866054,thing,1,/market/product/3866054,180.00,180.00,USD,U.S. dollars,$,new,...,1,3677448,CarsonChandler,United States,Christiansburg,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
2,3854476,thing,1,/market/product/3854476,41.99,41.99,USD,U.S. dollars,$,likenew,...,1,215794,Eric Engelmann,United States,Rockville,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
3,3853310,thing,1,/market/product/3853310,66.00,66.00,USD,Euros,€,new,...,1,1613268,JillySilly,Belgium,NaN,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
4,3845442,thing,1,/market/product/3845442,50.00,50.00,USD,U.S. dollars,$,verygood,...,1,38240,caesarmom,United States,Lorton,Die Macher (German-only second edition),/boardgameversion/24939/german-only-second-edi...,Die Macher,/boardgame/1/die-macher,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178007,2589638,thing,99992,/market/product/2589638,11.00,11.00,USD,Euros,€,new,...,1,9646,Edgar Gallego,Spain,Sarriguren,NaN,NaN,Kill Shot: The Counter-Terrorism Party Game,/boardgame/99992/kill-shot-the-counter-terrori...,NaN
178008,2545360,thing,99992,/market/product/2545360,5.00,5.00,USD,U.S. dollars,$,likenew,...,1,840122,ohkoolnerd,United States,Struthers,Kill Shot: The Counter-Terrorism Party Game (E...,/boardgameversion/101405/english-edition,Kill Shot: The Counter-Terrorism Party Game,/boardgame/99992/kill-shot-the-counter-terrori...,NaN
178009,2136397,thing,99992,/market/product/2136397,8.00,8.00,USD,U.S. dollars,$,likenew,...,1,102667,Strom40,United States,Ypsilanti,Kill Shot: The Counter-Terrorism Party Game (E...,/boardgameversion/101405/english-edition,Kill Shot: The Counter-Terrorism Party Game,/boardgame/99992/kill-shot-the-counter-terrori...,NaN
178010,1372069,thing,99992,/market/product/1372069,13.00,13.00,USD,U.S. dollars,$,new,...,1,681464,biphead1,United States,Queens,NaN,NaN,Kill Shot: The Counter-Terrorism Party Game,/boardgame/99992/kill-shot-the-counter-terrori...,NaN
